# Appendix A — PyTorch Foundations for Building LLMs from Scratch

This notebook is an annotated study version of the PyTorch appendix work for Sebastian Raschka's *Build a Large Language Model (From Scratch)*. The goal of this appendix is not to build the full language model yet; instead, it prepares the practical PyTorch skills that the rest of the project depends on.

In the main LLM chapters, tensors represent token IDs, embeddings, attention scores, logits, losses, gradients, and model parameters. Before implementing token embeddings, self-attention, transformer blocks, pretraining loops, and fine-tuning pipelines, this notebook builds the foundation through small, inspectable examples.

## What this notebook covers

1. **Tensor basics** — scalar, vector, matrix, and higher-dimensional tensors; dtype conversion; shape inspection; reshaping; transposition; and matrix multiplication.
2. **Automatic differentiation** — tracking parameters with `requires_grad=True`, computing a binary cross-entropy loss, extracting gradients manually with `torch.autograd.grad`, and using `.backward()`.
3. **Neural network modules** — defining a model with `torch.nn.Module`, stacking layers with `torch.nn.Sequential`, counting trainable parameters, running forward passes, and converting logits to probabilities.
4. **Datasets and DataLoaders** — wrapping raw tensors in a custom `Dataset`, batching examples, shuffling batches, and using `drop_last=True` for stable batch sizes.
5. **Training loop mechanics** — forward pass, loss computation, zeroing old gradients, backpropagation, optimizer step, training/evaluation modes, and accuracy calculation.
6. **Saving and loading models** — storing only the `state_dict`, then re-creating the model architecture and loading the learned weights.
7. **Device handling** — moving tensors and models between CPU and GPU, avoiding device mismatch errors, and writing CUDA-safe code that still runs on CPU-only machines.
8. **Distributed training sketch** — outlining the structure of PyTorch Distributed Data Parallel (DDP) for multi-GPU training, while keeping the notebook safe to run interactively.
9. **Matrix multiplication timing** — a small GPU benchmark that demonstrates why accelerators matter for deep learning workloads.

## Why this matters for the LLM project

A GPT-style model is mostly a composition of tensor operations, matrix multiplications, differentiable modules, and training loops. The same ideas used here on tiny toy data reappear later at larger scale: embeddings are lookup tensors, attention is batched matrix multiplication, transformer blocks are `nn.Module` objects, pretraining uses a loss/backward/step loop, and fine-tuning relies on careful dataloading and device placement.

The notebook intentionally keeps the examples small so each PyTorch concept can be understood before the complexity of the full LLM stack is added.


## 1. Imports and runtime setup

Start with the two libraries needed in this appendix: `torch` for tensor/deep-learning operations and `os` for environment variables used later by distributed training.

In [47]:
# PyTorch is the main framework used throughout this appendix and the LLM project.
import torch

# The os module is only needed later when configuring Distributed Data Parallel (DDP).
import os

## 2. Creating tensors with different ranks

PyTorch tensors generalize numbers, vectors, matrices, and higher-dimensional arrays. Later in the LLM project, token batches, embeddings, attention maps, and logits are all represented as tensors.

In [2]:
# Notes for this cell:
# - A 0D tensor is a scalar: one single value.
# - A 1D tensor is a vector.
# - A 2D tensor is a matrix.
# - A 3D tensor can be interpreted as a stack/batch of matrices.
# - Since all values are integers, PyTorch infers int64 by default.

tensor0d = torch.tensor(1)
tensor1d = torch.tensor([1, 2, 3])
tensor2d = torch.tensor([[1, 2],
                        [3, 4]])
tensor3d = torch.tensor([[[1, 2], [3, 4]],
                         [[5, 6], [7, 8]]])
print(tensor2d.dtype)

torch.int64


### Floating-point tensors

Neural networks usually work with floating-point values because gradients and optimization require continuous numbers. When a tensor is created from floats, PyTorch defaults to `float32`.

In [3]:
# Notes for this cell:
# - Float literals make PyTorch infer a floating-point dtype.

floatvec = torch.tensor([1.0, 2.0, 3.0])
print(floatvec.dtype)

torch.float32


### Converting tensor dtypes

Sometimes we need a different dtype for precision, memory, or compatibility. The `.to(...)` method returns a tensor converted to the requested dtype or device.

In [4]:
# Notes for this cell:
# - Convert the float32 vector to float64.

floatvec = floatvec.to(torch.float64)
print(floatvec.dtype)

torch.float64


### Inspecting tensor shape

Shape is one of the most important debugging tools in deep learning. Many LLM bugs come from mismatched dimensions, so checking `.shape` early is a good habit.

In [5]:
# Notes for this cell:
# - The 3D tensor contains 2 matrices, each with shape 2 x 2.

tensor3d.shape

torch.Size([2, 2, 2])

### Reshaping tensors

`reshape` and `view` can reorganize the same values into different dimensions. This is useful when flattening activations or preparing batches, but the total number of elements must stay unchanged.

In [6]:
# Notes for this cell:
# - Reshape the 2 x 2 x 2 tensor into a 4 x 2 tensor.
# - View the same 8 values as a 1 x 1 x 8 tensor.

print(tensor3d.reshape(4, 2), "\n", "-"*180)
print(tensor3d.view(1, 1, 8))

tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]]) 
 ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[[1, 2, 3, 4, 5, 6, 7, 8]]])


### Transposing dimensions

The `.T` attribute reverses dimensions. For 2D tensors this is the normal matrix transpose; for tensors with more than 2 dimensions, PyTorch reverses all dimensions, which is useful to inspect but should be used carefully.

In [7]:
# Notes for this cell:
# - For a 3D tensor, .T reverses all dimensions.
# - For a 2D tensor, .T is the familiar matrix transpose.

print(tensor3d.T, "\n", "-"*180)
print(tensor2d.T)

tensor([[[1, 5],
         [3, 7]],

        [[2, 6],
         [4, 8]]]) 
 ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
tensor([[1, 3],
        [2, 4]])


C:\Users\98922\AppData\Local\Temp\ipykernel_42124\159892877.py:1: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4424.)
  print(tensor3d.T, "\n", "-"*180)


### Matrix multiplication

Matrix multiplication is the core operation behind linear layers, attention scores, and projection layers in transformer models.

In [8]:
# Notes for this cell:
# - torch.matmul performs matrix multiplication.
# - With batched tensors, it applies matrix multiplication over the batch dimensions.

print(torch.matmul(tensor3d, tensor3d.T))
print(torch.matmul(tensor2d, tensor2d.T))

tensor([[[  7,  19],
         [ 15,  43]],

        [[ 34,  78],
         [ 46, 106]]])
tensor([[ 5, 11],
        [11, 25]])


### The `@` operator

PyTorch also supports Python's `@` operator as a compact form of matrix multiplication. This is often easier to read in notebooks.

In [9]:
# Notes for this cell:
# - The @ operator is equivalent to torch.matmul for this use case.

print(tensor3d @ tensor3d.T)

tensor([[[  7,  19],
         [ 15,  43]],

        [[ 34,  78],
         [ 46, 106]]])


## 3. Automatic differentiation with a tiny computation graph

Autograd records operations involving tensors that require gradients. Here we build a one-neuron binary classifier by hand: compute a linear score, apply sigmoid, calculate binary cross-entropy loss, and inspect the gradients.

In [10]:
# Notes for this cell:
# - Target label for one training example.
# - One input feature.
# - Learnable weight and bias. requires_grad=True tells PyTorch to track gradients.
# - Forward pass: linear transformation followed by sigmoid activation.
# - Binary cross-entropy compares the predicted probability with the true label.
# - torch.autograd.grad returns gradients explicitly without filling .grad by default.
# - retain_graph=True keeps the graph so we can still call backward() in the next cell.

import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


### Using `.backward()`

Training loops usually call `loss.backward()` instead of `torch.autograd.grad`. This accumulates gradients in each parameter's `.grad` attribute so an optimizer can update the parameters.

In [11]:
# Notes for this cell:
# - Backpropagate through the computation graph.
# - Gradients are now stored directly on the trainable tensors.

loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


## 4. Defining a small neural network module

PyTorch models usually subclass `torch.nn.Module`. This class registers layers and parameters automatically, which allows optimizers, saving/loading, and device movement to work cleanly.

In [12]:
class NeuralNetwork(torch.nn.Module):
    # A small fully connected neural network for classification examples.

    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        # Sequential applies the layers in the order listed here.
        self.layers = torch.nn.Sequential(
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # The final layer returns raw class scores, also called logits.
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        # Compute logits for the input batch.
        logits = self.layers(x)
        return logits

### Instantiating the model

This example creates a network that receives 50 input features and outputs 3 class logits. The printed structure is useful for checking layer sizes.

In [13]:
# Notes for this cell:
# - Create a model for a 3-class problem with 50 input features.

model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


### Counting trainable parameters

Parameter counting helps estimate model size. In LLMs, the same idea scales from hundreds of parameters here to millions or billions of parameters.

In [14]:
# Notes for this cell:
# - Count only parameters that require gradients.

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of model parameters: {num_params}")

Number of model parameters: 2213


### Inspecting a layer's weight matrix

A linear layer with 50 inputs and 30 outputs stores a weight matrix of shape `[30, 50]`. The output dimension comes first because each row corresponds to one output unit.

In [15]:
# Notes for this cell:
# - First Linear layer: out_features x in_features.

print(model.layers[0].weight.shape)

torch.Size([30, 50])


### Running a forward pass

A forward pass applies the model to input data. Setting a manual seed makes the random input reproducible.

In [16]:
# Notes for this cell:
# - Make the random example reproducible.
# - One example with 50 features.
# - The model returns raw logits, not probabilities.

torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)

tensor([[-0.1471,  0.0119, -0.3184]], grad_fn=<AddmmBackward0>)


### Disabling gradient tracking during inference

During evaluation or prediction, gradients are unnecessary. `torch.no_grad()` saves memory and computation by not building an autograd graph.

In [17]:
# Notes for this cell:
# - Inference does not need gradient tracking.

with torch.no_grad():
    out = model(X)

print(out)

tensor([[-0.1471,  0.0119, -0.3184]])


### Converting logits to probabilities

Classification models usually output logits. Applying `softmax` converts them into class probabilities that sum to 1 across the class dimension.

In [18]:
# Notes for this cell:
# - dim=1 applies softmax across the class dimension.

with torch.no_grad():
    out = torch.softmax(model(X), dim=1)

print(out)

tensor([[0.3317, 0.3889, 0.2795]])


## 5. Creating a toy classification dataset

The next examples use a tiny 2D dataset with two classes. It is intentionally simple so the mechanics of `Dataset`, `DataLoader`, and training loops stay visible.

In [19]:
# Five training examples with two features each.
X_train = torch.tensor([
    [-1.2,  3.1],
    [-0.9,  2.9],
    [-0.5,  2.6],
    [ 2.3, -1.1],
    [ 2.7, -1.5],
])
y_train = torch.tensor([0, 0, 0, 1, 1])

# Two test examples for a small evaluation check.
X_test = torch.tensor([
    [-0.8,  2.8],
    [ 2.6, -1.6],
])
y_test = torch.tensor([0, 1])

### Wrapping tensors in a custom `Dataset`

A `Dataset` defines how many examples exist and how to retrieve one example by index. This is the interface PyTorch's `DataLoader` expects.

In [20]:
from torch.utils.data import Dataset


class ToyDataset(Dataset):
    # A minimal dataset that returns one feature tensor and one label.

    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __len__(self):
        # Number of samples in the dataset.
        return self.labels.shape[0]

    def __getitem__(self, idx):
        # Return one example and its target label.
        one_x = self.features[idx]
        one_y = self.labels[idx]
        return one_x, one_y

### Creating train and test datasets

Now the raw tensors can be treated as PyTorch datasets. The same pattern is used later for text datasets and tokenized LLM training examples.

In [21]:
train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)

### Checking dataset length

This calls the dataset's `__len__` method. It is a simple sanity check before building a data loader.

In [22]:
print(len(train_ds))

5


### Building DataLoaders

A `DataLoader` handles batching, optional shuffling, and iteration. For this small dataset, `num_workers=0` is the safest setting across notebooks and Windows environments.

In [23]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0,
)

### Inspecting mini-batches

Iterating over the loader shows what the training loop will receive: a batch of features and a batch of labels.

In [24]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.5000,  2.6000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


### Using `drop_last=True`

When the dataset size is not divisible by the batch size, the last batch can be smaller. `drop_last=True` removes that final smaller batch, which can be useful when code assumes fixed batch shapes.

In [25]:
train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    num_workers=0,
    drop_last=True,
)

### Verifying fixed-size batches

With `drop_last=True`, the final one-example batch is skipped, so every training batch has exactly two examples.

In [26]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx+1}: ", x, y)

Batch 1:  tensor([[-0.5000,  2.6000],
        [ 2.3000, -1.1000]]) tensor([0, 1])
Batch 2:  tensor([[-1.2000,  3.1000],
        [ 2.7000, -1.5000]]) tensor([0, 1])


## 6. Training the small neural network

This is the standard PyTorch training-loop pattern: set train mode, run a forward pass, compute the loss, clear previous gradients, backpropagate, and update parameters with the optimizer.

In [27]:
# Notes for this cell:
# - Fix the random seed so the model initialization and shuffled batches are reproducible.
# - For this toy dataset, the model has 2 inputs and 2 output classes.
# - Stochastic Gradient Descent updates model parameters using the computed gradients.
# - Forward pass: compute class logits.
# - Cross-entropy combines log-softmax and negative log likelihood internally.
# - Gradients accumulate by default, so clear old gradients first.
# - Backpropagation computes gradients for all trainable parameters.
# - Optimizer step updates parameters using the current gradients.

import torch.nn.functional as F
torch.manual_seed(28)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        ##LOGGING
        print(f"Epoch: {epoch+1}/{num_epochs:03d}"
              f" | Batch {batch_idx}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

Epoch: 1/003 | Batch 0/002 | Train Loss: 0.75
Epoch: 1/003 | Batch 1/002 | Train Loss: 0.39
Epoch: 2/003 | Batch 0/002 | Train Loss: 0.31
Epoch: 2/003 | Batch 1/002 | Train Loss: 0.07
Epoch: 3/003 | Batch 0/002 | Train Loss: 0.04
Epoch: 3/003 | Batch 1/002 | Train Loss: 0.01


### Recounting parameters after changing the input/output sizes

The same architecture now has fewer parameters because it receives only 2 input features and predicts 2 classes.

In [28]:
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Number of model parameters: {num_params}")

Number of model parameters: 752


## 7. Evaluating model outputs

Switching to evaluation mode disables training-specific behavior such as dropout or batch-normalization updates. This model does not use those layers, but the habit is important.

In [29]:
model.eval()
with torch.no_grad():
    outputs = model(X_train)
print(outputs)

tensor([[ 1.9732, -2.4759],
        [ 1.8450, -2.3465],
        [ 1.6469, -2.1403],
        [-2.3899,  1.6433],
        [-2.8839,  2.0321]])


### Turning logits into readable probabilities

Softmax makes the model's confidence easier to inspect. This is for interpretation; `F.cross_entropy` expects raw logits during training.

In [30]:
# Notes for this cell:
# - Print probabilities in regular decimal notation instead of scientific notation.

torch.set_printoptions(sci_mode=False)
probs = torch.softmax(outputs, dim=1)
print(probs)

tensor([[0.9884, 0.0116],
        [0.9851, 0.0149],
        [0.9778, 0.0222],
        [0.0174, 0.9826],
        [0.0073, 0.9927]])


### Predicting classes from probabilities

The predicted class is the index with the largest probability for each example.

In [31]:
preds = torch.argmax(probs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


### Predicting classes directly from logits

Softmax does not change the ordering of class scores, so `argmax` on logits gives the same class predictions as `argmax` on probabilities.

In [32]:
preds = torch.argmax(outputs, dim=1)
print(preds)

tensor([0, 0, 0, 1, 1])


### Comparing predictions with labels

A boolean comparison shows which predictions are correct.

In [33]:
preds == y_train

tensor([True, True, True, True, True])

### Counting correct predictions

Summing the boolean tensor counts `True` values because `True` behaves like 1 and `False` behaves like 0.

In [34]:
torch.sum(preds == y_train).item()

5

### A reusable accuracy function

This helper works on CPU or GPU. It moves each batch to the requested device, runs inference without gradients, and returns the fraction of correct predictions.

In [60]:
def compute_accuracy(model, dataloader, device=None):
    # Compute classification accuracy for a model and dataloader.

    if device is None:
        # Infer the model's current device from its parameters.
        device = next(model.parameters()).device

    correct = 0
    total_examples = 0

    model.eval()

    with torch.no_grad():
        for features, labels in dataloader:
            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)
            preds = torch.argmax(logits, dim=1)

            correct += torch.sum(preds == labels).item()
            total_examples += labels.shape[0]

    return correct / total_examples

### Training accuracy

This evaluates the model on the same data it trained on. With such a tiny and separable dataset, perfect accuracy is expected after a few updates.

In [36]:
print(f"train accuracy: {compute_accuracy(model, train_loader)}")

train accuracy: 1.0


### Test accuracy

The test set is extremely small, so this number is only a demonstration of the evaluation workflow, not a reliable estimate of generalization.

In [37]:
print(f"test accuracy: {compute_accuracy(model, test_loader)}")

test accuracy: 1.0


## 8. Saving and loading model weights

The recommended PyTorch pattern is to save the model's `state_dict`, which contains learned parameters but not the full Python class definition.

In [38]:
MODEL_PATH = "pytorch_appendix.pth"

# Save learned parameters to disk.
torch.save(model.state_dict(), MODEL_PATH)

### Restoring a model from `state_dict`

To load the weights, first recreate the same architecture, then load the saved parameter dictionary. `map_location="cpu"` makes loading safe even on machines without CUDA.

In [39]:
# Notes for this cell:
# - Recreate the same architecture before loading the weights.
# - map_location keeps this cell portable across CPU-only and GPU machines.

model = NeuralNetwork(2, 2)
model.load_state_dict(torch.load("pytorch_appendix.pth"))

<All keys matched successfully>

## 9. Checking CUDA availability

GPU acceleration is important for deep learning, but notebooks should still run on CPU-only machines. This check controls the CUDA-specific examples below.

In [40]:
print(torch.cuda.is_available())

True


### CPU tensor addition

Operations work normally when all tensors are on the same device. Here both tensors live on the CPU.

In [41]:
tens1 = torch.tensor([1.0, 2.0, 3.0])
tens2 = torch.tensor([1.0, 2.0, 3.0])
print(tens1 + tens2)

tensor([2., 4., 6.])


### GPU tensor addition

When CUDA is available, tensors can be moved to the GPU. The operation is the same, but the data lives on the CUDA device.

In [42]:
# Notes for this cell:
# - This cell creates two tensors directly on the GPU and adds them.
# - It requires CUDA; the saved output was produced on a CUDA-enabled run.

tens1 = torch.tensor([1.0, 2.0, 3.0]).to("cuda")
tens2 = torch.tensor([1.0, 2.0, 3.0]).to("cuda")
print(tens1 + tens2)

tensor([2., 4., 6.], device='cuda:0')


### Understanding device mismatch errors

PyTorch operations require tensors to be on the same device. This cell intentionally demonstrates the error safely with `try/except` instead of stopping the notebook.

In [43]:
# Notes for this cell:
# - This fails because one tensor is on CPU and the other is on CUDA.

tens1 = tens1.to("cpu")
print(tens1 + tens2)

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!

## 10. Training on the selected device

The safe pattern is to choose `cuda` when available and otherwise fall back to `cpu`. The model and each batch must be moved to the same device.

In [44]:
# Notes for this cell:
# - This repeats the training loop after moving the model and each mini-batch to CUDA.
# - The saved output was produced on a machine where CUDA was available.

torch.manual_seed(28)
model = NeuralNetwork(num_inputs=2, num_outputs=2)
device = torch.device("cuda")
model = model.to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):
        features, labels = features.to(device), labels.to(device)
        logits = model(features)
        loss = F.cross_entropy(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    ##LOGGING
        print(f"Epoch: {epoch+1}/{num_epochs:03d}"
              f" | Batch {batch_idx}/{len(train_loader):03d}"
              f" | Train Loss: {loss:.2f}")

Epoch: 1/003 | Batch 0/002 | Train Loss: 0.75
Epoch: 1/003 | Batch 1/002 | Train Loss: 0.39
Epoch: 2/003 | Batch 0/002 | Train Loss: 0.31
Epoch: 2/003 | Batch 1/002 | Train Loss: 0.07
Epoch: 3/003 | Batch 0/002 | Train Loss: 0.04
Epoch: 3/003 | Batch 1/002 | Train Loss: 0.01


### Saving the active device object

This compact expression is the standard way to define a reusable device variable across notebooks and scripts.

In [45]:
# Select CUDA when available; otherwise use CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 11. Distributed Data Parallel imports

Distributed Data Parallel (DDP) is PyTorch's recommended pattern for multi-GPU training. The next cells define the structure, but the actual launch is disabled in the notebook because DDP is more reliable when run from a Python script.

In [46]:
import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group

### Setting up a DDP process group

Each GPU usually runs one process. The process group lets those processes communicate and synchronize gradients during training.

In [48]:
def ddp_setup(rank, world_size):
    # Initialize the distributed process group for one worker process.

    os.environ["MASTER_ADDR"] = "localhost"
    os.environ["MASTER_PORT"] = "12345"

    # NCCL is preferred for CUDA GPUs; Gloo is a CPU-compatible fallback.
    backend = "nccl" if torch.cuda.is_available() else "gloo"

    init_process_group(
        backend=backend,
        rank=rank,
        world_size=world_size,
    )

    if torch.cuda.is_available():
        torch.cuda.set_device(rank)

### Preparing distributed dataloaders

`DistributedSampler` splits the dataset across worker processes so each process sees a different slice of the data.

In [51]:
def prepare_dataset():
    # Create DataLoaders that use DistributedSampler for DDP.

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=2,
        shuffle=False,             # DistributedSampler controls shuffling.
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=True,
        sampler=DistributedSampler(train_ds),
    )

    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=2,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
        drop_last=False,
        sampler=DistributedSampler(test_ds, shuffle=False),
    )

    return train_loader, test_loader

### DDP training function

This function is what each process runs. It creates the model on the correct device, wraps it with `DDP`, trains on that process's data shard, evaluates, and then destroys the process group.

In [54]:
def main(rank, world_size, num_epochs):
    # Train the toy model with Distributed Data Parallel.

    ddp_setup(rank, world_size)
    train_loader, test_loader = prepare_dataset()

    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")

    model = NeuralNetwork(num_inputs=2, num_outputs=2).to(device)

    # For CUDA DDP, each process is bound to one GPU.
    ddp_device_ids = [rank] if torch.cuda.is_available() else None
    model = DDP(model, device_ids=ddp_device_ids)

    optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

    for epoch in range(num_epochs):
        model.train()

        # Ensures different shuffling across epochs in distributed training.
        if hasattr(train_loader.sampler, "set_epoch"):
            train_loader.sampler.set_epoch(epoch)

        for features, labels in train_loader:
            features = features.to(device)
            labels = labels.to(device)

            logits = model(features)
            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            print(
                f"[Rank {rank}] Epoch: {epoch + 1:03d}/{num_epochs:03d}"
                f" | Batch size: {labels.shape[0]:03d}"
                f" | Train Loss: {loss:.2f}"
            )

    # model.module is the original NeuralNetwork wrapped by DDP.
    train_acc = compute_accuracy(model.module, train_loader, device=device)
    test_acc = compute_accuracy(model.module, test_loader, device=device)

    print(f"[Rank {rank}] Training Accuracy: {train_acc:.3f}")
    print(f"[Rank {rank}] Test Accuracy: {test_acc:.3f}")

    destroy_process_group()

### DDP launch guard

Launching DDP from inside notebooks is fragile, especially on Windows. Keep this disabled here and use the included `multiple_gpus.py` script when you want to run the multi-GPU version from the terminal.

In [59]:
# DDP is safer to launch from a terminal script than from inside a notebook.
# Keep this disabled here and use multiple_gpus.py for the runnable version.
RUN_DDP_DEMO = False

if RUN_DDP_DEMO:
    print(f"Number GPUs available on Device: {torch.cuda.device_count()}")
    torch.manual_seed(28)
    num_epochs = 3
    world_size = torch.cuda.device_count()
    mp.spawn(main, args=(world_size, num_epochs), nprocs=world_size)

## 12. Matrix multiplication benchmark setup

Large language models spend a lot of time in matrix multiplications. This small benchmark allocates a square matrix on the GPU when CUDA is available.

In [ ]:
# Allocate the first large matrix on the GPU for a matrix-multiplication benchmark.
# This cell requires CUDA and enough available GPU memory.
a = torch.randn(10000, 10000).to("cuda")

### Allocating the second benchmark matrix

The second matrix has the same shape as the first. Multiplying `a @ b` will perform a large dense matrix multiplication on the GPU.

In [38]:
# Allocate the second large matrix on the GPU.
b = torch.randn(10000, 10000).to("cuda")

### Timing one GPU matrix multiplication

CUDA operations are asynchronous, so `torch.cuda.synchronize()` is used before and after timing to measure the actual computation time more accurately.

In [ ]:
# Time a large GPU matrix multiplication in Jupyter.
%timeit a @ b